# Genetic Algorithm for Warehouse Order-Picking Optimisation
---

## Research Overview

This notebook implements a **Genetic Algorithm (GA)** metaheuristic to solve the  
**Warehouse Order-Picking Problem** — a variant of the Vehicle Routing Problem (VRP).

### Problem Statement
Given a warehouse with a fixed shelf layout, we want to find the optimal  
**assignment of SKUs (Stock-Keeping Units) to shelf locations** such that the  
total distance (and time) an order-picker must travel to fulfil a batch of  
orders is minimised.

### Algorithm Pipeline
1. **Reorder-Rate Generation** — Simulate realistic SKU demand frequencies  
2. **Order Generation** — Sample orders from the demand distribution  
3. **Location Point Generation** — Encode the 3-D warehouse shelf geometry  
4. **Initial Population** — Create random SKU-to-location assignments  
5. **Fitness Evaluation** — Compute total travel distance + retrieval time  
6. **Selection → Crossover → Mutation** — Evolve the population  
7. **Convergence** — Repeat for *N* generations; track best solution  

### Routing Strategies Compared
| Strategy | Description |
|---|---|
| **S-Shape** | Traverse each aisle fully in a snake pattern |
| **Return** | Enter each aisle only as far as needed, then return |
| **Mid-Point** | Split aisles at the midpoint; approach from nearest end |

### Storage Policies Compared
| Policy | Description |
|---|---|
| **W (Random)** | SKUs assigned randomly to any shelf location |
| **GZA (Gold-Zone Assignment)** | High-demand SKUs assigned to ergonomic mid-level positions |


## 1. Imports & Global Configuration

In [ ]:
import random
import hashlib
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import PatternFill

# ── Reproducibility ────────────────────────────────────────────────────────────
# Seeds are set per-experiment (see Section 2) to allow reproducible runs
# without locking the entire module to a single seed.

# ── Warehouse Physical Parameters ──────────────────────────────────────────────
NUM_SHELVES: int = 5       # Total rack structures in the warehouse
NUM_LEVELS: int = 4        # Vertical levels per rack (0 = ground, 3 = top)
NUM_COLUMNS: int = 25      # Horizontal slots per rack face
AISLE_LENGTH: float = 25.0 # Assumed aisle depth in metres
AISLE_SPACING: float = 10.0 # Centre-to-centre distance between rack rows (m)
COLUMN_SPACING: float = 10.0 # Distance between shelf columns (m)

# ── GA Hyper-parameters (defaults; overridden in experiments) ──────────────────
DEFAULT_POP_SIZE: int = 200
DEFAULT_GENERATIONS: int = 200
DEFAULT_MUTATION_RATE: float = 0.5

# ── Time Penalties per Vertical Level (seconds) ───────────────────────────────
# The order picker must stoop, reach, or use a step-stool depending on the
# shelf level.  These values capture the ergonomic retrieval cost.
LEVEL_TIME: Dict[int, float] = {
    0: 2.0 * 60,    # Ground level  — requires bending; slowest
    1: 0.5 * 60,    # Lower-mid     — golden zone; fastest
    2: 0.5 * 60,    # Upper-mid     — golden zone; fastest
    3: 2.5 * 60,    # Top level     — requires stool/elevator; slowest
}


## 2. Demand Simulation & Order Generation

We model SKU demand using a **tri-modal distribution** to reflect the typical
80/20 (Pareto) pattern seen in real warehouses:

- **Top 100 SKUs** — high-velocity (reorder rate 6–10)  
- **Next 200 SKUs** — medium-velocity (reorder rate 3–4.5)  
- **Remaining 200 SKUs** — slow-movers (reorder rate 0.1–2.5)  

Orders are then sampled proportional to reorder rate, mimicking realistic
demand-weighted picking.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1  Reorder-Rate Distribution
# ─────────────────────────────────────────────────────────────────────────────
# Each SKU is assigned a reorder rate drawn from one of three uniform bands.
# We sort descending so that SKU 0 is always the fastest mover, which makes
# gold-zone assignment (Section 3) straightforward.
# ─────────────────────────────────────────────────────────────────────────────

def generate_reorder_rates(
    n_fast: int = 100,
    n_medium: int = 200,
    n_slow: int = 200,
    seed: int = 30,
) -> List[float]:
    """Generate a sorted (descending) list of SKU reorder rates.

    Parameters
    ----------
    n_fast   : Number of fast-moving SKUs (high reorder rate band).
    n_medium : Number of medium-velocity SKUs.
    n_slow   : Number of slow-moving SKUs.
    seed     : Random seed for reproducibility.

    Returns
    -------
    combined_list : Sorted list of reorder rates, length = n_fast + n_medium + n_slow.
    """
    random.seed(seed)

    # Fast movers — high reorder rate
    fast = [round(random.uniform(6.0, 10.0), 2) for _ in range(n_fast)]

    # Medium movers
    medium = [round(random.uniform(3.0, 4.5), 2) for _ in range(n_medium)]

    # Slow movers
    slow = [round(random.uniform(0.1, 2.5), 2) for _ in range(n_slow)]

    combined = fast + medium + slow
    combined.sort(reverse=True)   # Highest reorder rate → SKU 0
    return combined


reorder_rates = generate_reorder_rates(seed=30)

# Build the SKU → reorder-rate lookup table
sku_reorder_rate: Dict[int, float] = {
    idx: rate for idx, rate in enumerate(reorder_rates)
}
sku_ids: List[int] = list(range(len(reorder_rates)))

# ── Visualisation ──────────────────────────────────────────────────────────────
plt.figure(figsize=(14, 4))
plt.plot(reorder_rates, linewidth=1.2)
plt.xlabel("SKU ID")
plt.ylabel("Reorder Rate")
plt.title("Reorder Rate Distribution of SKUs (seed=30)")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Total SKUs: {len(sku_ids)}")
print(f"Reorder rate range: [{min(reorder_rates):.2f}, {max(reorder_rates):.2f}]")


### 2.2 Order Generation

Each of the 1 000 simulated orders contains between 2 and 50 distinct SKUs,
drawn with probability proportional to the reorder rate.  Using `random.choices`
with explicit weights ensures high-velocity SKUs appear in orders more
frequently, replicating real warehouse demand.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.2  Simulate 1 000 Customer Orders
# ─────────────────────────────────────────────────────────────────────────────

def generate_orders(
    sku_ids: List[int],
    sku_reorder_rate: Dict[int, float],
    n_orders: int = 1000,
    max_order_size: int = 50,
    seed: int = 30,
) -> List[List[int]]:
    """Simulate a batch of customer orders using demand-weighted sampling.

    Each order is a *set* of unique SKU IDs (no duplicate picks per order).

    Parameters
    ----------
    sku_ids          : All available SKU identifiers.
    sku_reorder_rate : Mapping of SKU ID → demand weight.
    n_orders         : Number of orders to generate.
    max_order_size   : Upper bound on SKUs per order (sampled uniformly in [2, max_order_size]).
    seed             : Random seed.

    Returns
    -------
    orders : List of orders; each order is a list of unique SKU IDs.
    """
    random.seed(seed)
    weights = [sku_reorder_rate[s] for s in sku_ids]
    orders: List[List[int]] = []

    for _ in range(n_orders):
        order_size = random.choice(range(2, max_order_size + 1))
        # Sample with replacement then deduplicate to get unique SKUs per order
        sampled = random.choices(sku_ids, weights=weights, k=order_size)
        orders.append(list(set(sampled)))

    return orders


orders: List[List[int]] = generate_orders(sku_ids, sku_reorder_rate, seed=30)

# Keep an immutable copy for the time-calculation phase (prevents accidental mutation)
orders_ref: List[List[int]] = [o[:] for o in orders]

# ── Frequency Distribution Visualisation ──────────────────────────────────────
all_picks = [sku for order in orders for sku in order]
counter = Counter(all_picks)

plt.figure(figsize=(15, 5))
plt.bar(counter.keys(), counter.values(), color="steelblue", width=0.8)
plt.xlabel("SKU ID")
plt.ylabel("Frequency in Orders")
plt.title("SKU Frequency Distribution Across 1 000 Generated Orders (seed=30)")
plt.xticks(range(0, len(sku_ids) + 1, 25))
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()

print(f"Orders generated: {len(orders)}")
print(f"Avg SKUs per order: {np.mean([len(o) for o in orders]):.1f}")


## 3. Warehouse Geometry & Location Assignment

The warehouse is modelled as a 3-D grid:

- **x-axis** — cross-aisle direction (which rack row)  
- **y-axis** — along-aisle direction (which column within a rack)  
- **z-axis** — vertical level (0 = ground … 3 = top)  

The **depot/pickup point** is at the origin `(0, 0)`.  
Rack rows start at x = 10 m and are spaced 20 m apart  
(10 m for the rack itself + 10 m aisle on each side).

Two storage policies are implemented:

| Policy | `z`-levels used | Description |
|---|---|---|
| **W** (Random) | 0–3 | Uniform assignment; baseline |
| **GZA** (Gold-Zone Assignment) | Gold: 1–2  /  Non-gold: 0, 3 | High-demand SKUs → ergonomic levels |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3.1  3-D Location Point Generators
# ─────────────────────────────────────────────────────────────────────────────

def generate_3d_locations(
    num_shelves: int,
    num_levels: int,
    num_columns: int,
    aisle_spacing: float = AISLE_SPACING,
    column_spacing: float = COLUMN_SPACING,
) -> List[Tuple[float, float, int]]:
    """Generate all (x, y, z) shelf positions for a regular rack layout.

    The depot is implicitly at (0, 0, _).  Rack rows are placed at
    x = aisle_spacing, 3*aisle_spacing, 5*aisle_spacing, … so that
    consecutive rows are separated by a full aisle width.

    Parameters
    ----------
    num_shelves   : Number of rack structures.
    num_levels    : Number of vertical levels per rack (z goes from 0 to num_levels-1).
    num_columns   : Number of horizontal slots per rack face.
    aisle_spacing : Rack centre-to-wall distance (metres).
    column_spacing: Distance between adjacent columns (metres).

    Returns
    -------
    List of (x, y, z) tuples, one per shelf slot.
    """
    locations: List[Tuple[float, float, int]] = []
    # Step by 2 so x values are 1*spacing, 3*spacing, … (odd multiples)
    for shelf_idx in range(1, num_shelves * 2, 2):
        x = shelf_idx * aisle_spacing
        for col in range(num_columns):
            y = col * column_spacing
            for level in range(num_levels):
                locations.append((x, y, level))
    return locations


def generate_2d_locations(
    num_shelves: int,
    num_columns: int,
    aisle_spacing: float = AISLE_SPACING,
    column_spacing: float = COLUMN_SPACING,
) -> List[Tuple[float, float]]:
    """Generate (x, y) positions for distance-matrix computation (ignores z).

    The 2-D projection is used by all routing algorithms because horizontal
    travel is the dominant distance component.

    Returns
    -------
    List of (x, y) tuples, one per unique shelf face position.
    """
    locations: List[Tuple[float, float]] = []
    for shelf_idx in range(1, num_shelves * 2, 2):
        x = shelf_idx * aisle_spacing
        for col in range(num_columns):
            y = col * column_spacing
            locations.append((x, y))
    return locations


# ─────────────────────────────────────────────────────────────────────────────
# 3.2  Gold-Zone Assignment (GZA) Location Pools
# ─────────────────────────────────────────────────────────────────────────────

def generate_gold_zone_locations(
    num_shelves: int,
    num_columns: int,
    aisle_spacing: float = AISLE_SPACING,
    column_spacing: float = COLUMN_SPACING,
) -> List[Tuple[float, float, int]]:
    """Generate locations at the ergonomic 'golden' levels (z=1 and z=2).

    Golden levels are mid-height shelves that require the least physical effort
    to pick from.  High-demand SKUs are assigned here under the GZA policy.
    """
    locations: List[Tuple[float, float, int]] = []
    # num_levels is fixed to 2 for golden zone (levels 1 and 2)
    gold_levels = [1, 2]
    for shelf_idx in range(1, num_shelves * 2, 2):
        x = shelf_idx * aisle_spacing
        for col in range(num_columns):
            y = col * column_spacing
            for z in gold_levels:
                locations.append((x, y, z))
    return locations


def generate_non_gold_zone_locations(
    num_shelves: int,
    num_levels: int,
    num_columns: int,
    aisle_spacing: float = AISLE_SPACING,
    column_spacing: float = COLUMN_SPACING,
) -> List[Tuple[float, float, int]]:
    """Generate locations at non-ergonomic levels (z=0 and z=3).

    Slow-moving SKUs are stored here under the GZA policy.
    The range step of 3 selects z=0 (ground) and z=3 (top).
    """
    locations: List[Tuple[float, float, int]] = []
    non_gold_levels = range(0, num_levels, 3)   # selects 0 and 3
    for shelf_idx in range(1, num_shelves * 2, 2):
        x = shelf_idx * aisle_spacing
        for col in range(num_columns):
            y = col * column_spacing
            for z in non_gold_levels:
                locations.append((x, y, z))
    return locations


# ─────────────────────────────────────────────────────────────────────────────
# 3.3  Build Active Location Dictionaries
# ─────────────────────────────────────────────────────────────────────────────
# Two lookup dicts are needed:
#   sku_location_3d  — maps slot index → (x, y, z)   used for time calculation
#   sku_location_2d  — maps slot index → (x, y)      used for distance routing
# ─────────────────────────────────────────────────────────────────────────────

# Gold-zone (GZA) 3-D locations: high-demand slots first, then non-gold
_gold_locations = generate_gold_zone_locations(NUM_SHELVES, NUM_COLUMNS)
_non_gold_locations = generate_non_gold_zone_locations(
    NUM_SHELVES, NUM_LEVELS, NUM_COLUMNS
)
gza_locations_3d = _gold_locations + _non_gold_locations

# Active 3-D mapping (used throughout evaluation; switch to a different list
# to compare W vs GZA simply by reassigning this variable)
sku_location_3d: Dict[int, Tuple[float, float, int]] = {
    idx: loc for idx, loc in enumerate(gza_locations_3d)
}

# 2-D mapping for routing distance computation
_2d_locations = generate_2d_locations(NUM_SHELVES, NUM_COLUMNS)
sku_location_2d: Dict[int, Tuple[float, float]] = {
    idx: loc for idx, loc in enumerate(_2d_locations)
}

print(f"Total 3-D shelf slots : {len(sku_location_3d)}")
print(f"  └─ Gold-zone slots  : {len(_gold_locations)}")
print(f"  └─ Non-gold slots   : {len(_non_gold_locations)}")
print(f"Total 2-D positions   : {len(sku_location_2d)}")
print(f"Sample 3-D entries    : {list(sku_location_3d.items())[:4]}")


## 4. Routing Algorithms

Three classical warehouse routing heuristics are implemented.  Each takes a
sorted list of 2-D coordinates (the locations of SKUs in one order) and returns
the estimated travel distance in metres.

> **Note:** All routing algorithms work on the *2-D projection* `(x, y)`.  
> Vertical (z) movement cost is captured separately in the time model (Section 6).


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1  S-Shape Routing
# ─────────────────────────────────────────────────────────────────────────────
# The order picker traverses aisles in a snake/serpentine pattern:
#   - Enter every aisle that contains at least one pick location.
#   - Traverse the full aisle length, alternating direction each time.
# This strategy is simple to implement and works well when picks are spread
# across many aisles, but it wastes distance in partially-filled aisles.
# ─────────────────────────────────────────────────────────────────────────────

def sort_s_shape(
    coordinates: List[Tuple[float, float]]
) -> List[Tuple[float, float]]:
    """Sort pick coordinates into S-shape traversal order.

    Algorithm:
    1. Sort all coordinates by x (aisle/row index).
    2. Within each aisle, sort by y ascending for even-indexed aisles
       and y descending for odd-indexed aisles — creating the 'snake'.

    Parameters
    ----------
    coordinates : List of (x, y) pick positions for a single order.

    Returns
    -------
    Sorted list of (x, y) coordinates in S-shape traversal order.
    """
    # Step 1: sort by row (x-coordinate)
    coordinates.sort(key=lambda c: c[0])

    # Identify the unique aisle x-values in visit order
    unique_x = sorted(set(c[0] for c in coordinates))

    # Step 2: group picks by aisle
    groups: List[List[Tuple[float, float]]] = []
    current_group: List[Tuple[float, float]] = []
    prev_x = None
    for coord in coordinates:
        if coord[0] != prev_x:
            if current_group:
                groups.append(current_group)
            current_group = []
            prev_x = coord[0]
        current_group.append(coord)
    if current_group:
        groups.append(current_group)

    # Step 3: within each aisle, reverse direction for odd-indexed aisles
    for group in groups:
        aisle_index = unique_x.index(group[0][0])
        if aisle_index % 2 == 0:
            group.sort(key=lambda c: c[1])           # ascending  (enter from bottom)
        else:
            group.sort(key=lambda c: c[1], reverse=True)  # descending (enter from top)

    # Step 4: flatten back to a single sequence
    return [coord for group in groups for coord in group]


def s_shape_distance(sorted_order: List[Tuple[float, float]]) -> float:
    """Compute total travel distance for an order under S-shape routing.

    The total distance consists of:
      - Distance from depot (0,0) to the first pick location
      - Full traversal of all intermediate aisles (vertical movement)
      - Horizontal hops between consecutive pick locations
      - Return from the last pick to the depot

    Two sub-cases exist:
      - Even number of aisles visited: the picker exits the last aisle at the
        same end it entered, so return distance = x-coordinate only.
      - Odd number of aisles visited: the picker must backtrack from the
        deepest point in the last aisle, so return adds both x and y.

    Parameters
    ----------
    sorted_order : Pick coordinates already sorted by `sort_s_shape`.

    Returns
    -------
    Total estimated travel distance in metres.
    """
    if not sorted_order:
        return 0.0

    unique_aisles = sorted(set(c[0] for c in sorted_order))
    n_aisles = len(unique_aisles)

    # Horizontal distance: sum of x-differences between consecutive picks
    horizontal_dist = sum(
        abs(sorted_order[i + 1][0] - sorted_order[i][0])
        for i in range(len(sorted_order) - 1)
    )

    # Distance from depot to first pick
    dist_depot_to_first = sorted_order[0][0] + sorted_order[0][1]

    if n_aisles % 2 == 0:
        # Even aisles: picker exits last aisle at the entrance side
        vertical_dist = n_aisles * AISLE_LENGTH
        dist_last_to_depot = sorted_order[-1][0]  # y already covered by vertical
    else:
        # Odd aisles: last aisle is only partially traversed; picker returns
        vertical_dist = (n_aisles - 1) * AISLE_LENGTH + sorted_order[-1][1]
        dist_last_to_depot = sorted_order[-1][0] + sorted_order[-1][1]

    return dist_depot_to_first + vertical_dist + horizontal_dist + dist_last_to_depot


### 4.2 Return Routing (commented — retained for reference)

In **return routing** the picker enters each aisle only as far as the deepest
pick in that aisle, then returns to the main aisle.  This is optimal when picks
are clustered near the aisle entrance but wastes distance for back-of-aisle picks.

The implementation is preserved below in commented form for comparison.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.2  Return Routing  (disabled — kept for experimental comparison)
# ─────────────────────────────────────────────────────────────────────────────
# Strategy: enter each aisle to the farthest pick, then return to the main
# corridor before entering the next aisle.
# ─────────────────────────────────────────────────────────────────────────────

# def return_routing(order_list: List[Tuple[float, float]]) -> float:
#     """Compute travel distance under return-routing strategy."""
#     order_list.sort(key=lambda c: c[0])
#
#     # Group picks by aisle (x-coordinate)
#     groups: List[List[Tuple[float, float]]] = []
#     current_group: List[Tuple[float, float]] = []
#     prev_x = None
#     for coord in order_list:
#         if coord[0] != prev_x:
#             if current_group:
#                 groups.append(current_group)
#             current_group = []
#             prev_x = coord[0]
#         current_group.append(coord)
#     if current_group:
#         groups.append(current_group)
#
#     # Sort each aisle's picks by y (ascending) to find the deepest easily
#     for group in groups:
#         group.sort(key=lambda c: c[1])
#
#     # Vertical movement: for each aisle, travel to the farthest pick and back
#     # (deepest_y * 2 per aisle)
#     total_vertical = sum(group[-1][1] * 2 for group in groups)
#
#     dist_depot_to_first = order_list[0][0]  # y handled by vertical
#     dist_last_to_depot = order_list[-1][0]  # y handled by vertical
#
#     horizontal_dist = sum(
#         abs(order_list[i + 1][0] - order_list[i][0])
#         for i in range(len(order_list) - 1)
#     )
#
#     return dist_depot_to_first + total_vertical + horizontal_dist + dist_last_to_depot


### 4.3 Mid-Point Routing (commented — retained for reference)

**Mid-point routing** splits each aisle at its midpoint.  Picks below the
midpoint are reached from the bottom entrance; picks above from the top entrance.
This reduces backtracking when picks are scattered throughout the aisle.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.3  Mid-Point Routing  (disabled — kept for experimental comparison)
# ─────────────────────────────────────────────────────────────────────────────
# Strategy: for each aisle, decide whether to enter from the bottom or the top
# based on where the picks lie relative to the aisle midpoint.
# ─────────────────────────────────────────────────────────────────────────────

# def mid_routing(order_list: List[Tuple[float, float]]) -> float:
#     """Compute travel distance under mid-point routing."""
#     MIDPOINT = AISLE_LENGTH / 2   # Half the aisle length
#
#     order_list.sort(key=lambda c: c[0])
#
#     groups: List[List[Tuple[float, float]]] = []
#     current_group: List[Tuple[float, float]] = []
#     prev_x = None
#     for coord in order_list:
#         if coord[0] != prev_x:
#             if current_group:
#                 groups.append(current_group)
#             current_group = []
#             prev_x = coord[0]
#         current_group.append(coord)
#     if current_group:
#         groups.append(current_group)
#
#     for group in groups:
#         group.sort(key=lambda c: c[1])
#
#     # Per-aisle vertical cost depends on whether the deepest pick is before
#     # or after the midpoint
#     total_vertical = 0.0
#     for group in groups:
#         deepest_y = group[-1][1]
#         if deepest_y < MIDPOINT:
#             total_vertical += deepest_y * 2   # Enter and return
#         else:
#             total_vertical += AISLE_LENGTH    # Traverse full aisle
#
#     dist_depot_to_first = order_list[0][0]
#     dist_last_to_depot = order_list[-1][0]
#
#     horizontal_dist = sum(
#         abs(order_list[i + 1][0] - order_list[i][0])
#         for i in range(len(order_list) - 1)
#     )
#
#     return dist_depot_to_first + total_vertical + horizontal_dist + dist_last_to_depot


## 5. Fitness Evaluation

The fitness of a **chromosome** (one SKU-to-location assignment) is computed as:

$$\text{fitness} = \underbrace{\sum_{o \in \text{orders}} d(o)}_\text{total travel distance}
                  + \underbrace{\sum_{o \in \text{orders}} \sum_{j \in o} t_{z(j)}}_\text{vertical retrieval time}$$

where:
- $d(o)$ = distance to fulfil order $o$ under the active routing strategy  
- $t_{z(j)}$ = time penalty for retrieving SKU $j$ from its vertical level $z$  

Lower fitness = better chromosome.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.1  Order Coordinate Resolver
# ─────────────────────────────────────────────────────────────────────────────

def resolve_order_coordinates(
    chromosome: List[int],
    order: List[int],
) -> List[Tuple[float, float]]:
    """Map a single order's SKU IDs to (x, y) coordinates given a chromosome.

    A chromosome is a permutation of slot indices: chromosome[i] = s means
    SKU s is stored in slot i.  The inverse mapping (SKU → slot) is derived
    on the fly via list.index().

    Parameters
    ----------
    chromosome : Current SKU-to-slot assignment (permutation of slot indices).
    order      : List of SKU IDs in the order.

    Returns
    -------
    Unique (x, y) coordinates for the order, preserving insertion order.
    """
    coords: List[Tuple[float, float]] = []
    seen = set()
    for sku in order:
        slot = chromosome.index(sku)           # which slot holds this SKU?
        xy = sku_location_2d[slot][:2]         # 2-D position of that slot
        if xy not in seen:
            coords.append(xy)
            seen.add(xy)
    return coords


# ─────────────────────────────────────────────────────────────────────────────
# 5.2  Vertical Retrieval Time Calculator
# ─────────────────────────────────────────────────────────────────────────────

def compute_retrieval_time(
    chromosome: List[int],
    order: List[int],
) -> float:
    """Compute the total vertical retrieval time for one order.

    For each SKU in the order, look up the z-level of its assigned slot and
    add the corresponding ergonomic time penalty (defined in LEVEL_TIME).

    Parameters
    ----------
    chromosome : Current SKU-to-slot assignment.
    order      : List of SKU IDs in the order.

    Returns
    -------
    Total retrieval time in seconds for this order.
    """
    total_time = 0.0
    for sku in order:
        slot = chromosome.index(sku)
        z_level = sku_location_3d[slot][2]     # vertical level of this slot
        total_time += LEVEL_TIME.get(z_level, 0.0)
    return total_time


# ─────────────────────────────────────────────────────────────────────────────
# 5.3  Population Fitness Evaluator
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_population(
    population: List[List[int]],
) -> Tuple[List[float], List[List[int]]]:
    """Evaluate the fitness of every chromosome in the population.

    For each chromosome:
      1. Resolve all orders to (x, y) coordinate sequences.
      2. Apply S-shape sort to each order's coordinates.
      3. Compute S-shape travel distance per order.
      4. Compute vertical retrieval time per order.
      5. Fitness = total distance + total retrieval time (lower is better).

    After fitness evaluation the function also runs selection (top-20 elitism)
    and crossover to produce a new offspring list.  This coupling follows the
    original implementation structure.

    Parameters
    ----------
    population : List of chromosomes (each a permutation of slot indices).

    Returns
    -------
    fitness_values : Fitness score for each chromosome.
    offspring      : New population generated by crossover over the elite set.
    """
    total_distances: List[float] = []   # one entry per chromosome
    total_times: List[float] = []       # one entry per chromosome

    for chromosome in population:
        # ── Distance component ─────────────────────────────────────────────
        dist_sum = 0.0
        for order in orders:
            coords = resolve_order_coordinates(chromosome, order)
            sorted_coords = sort_s_shape(coords)
            dist_sum += s_shape_distance(sorted_coords)
        total_distances.append(dist_sum)

        # ── Time component (vertical retrieval penalties) ──────────────────
        time_sum = 0.0
        for order in orders_ref:
            time_sum += compute_retrieval_time(chromosome, order)
        total_times.append(time_sum)

    # Combined fitness score
    fitness_values = [d + t for d, t in zip(total_distances, total_times)]

    # ── Summary statistics ────────────────────────────────────────────────
    min_fit = min(fitness_values)
    max_fit = max(fitness_values)
    print(f"  Fitness range: [{min_fit:.0f}, {max_fit:.0f}]  "
          f"Δ = {(max_fit - min_fit)/60:.1f} min")

    # ── Selection (top-20 elitism) ────────────────────────────────────────
    sorted_indices = sorted(range(len(fitness_values)), key=lambda i: fitness_values[i])
    elite = [population[i] for i in sorted_indices]

    # ── Crossover: produce a new full population ───────────────────────────
    offspring: List[List[int]] = elite[:20]   # carry elite forward unchanged
    for _ in range(len(population)):
        p1, p2 = random.sample(elite[:20], 2)
        child = order_crossover(p1, p2)
        offspring.append(child)

    return fitness_values, offspring


## 6. Genetic Algorithm Operators

### 6.1 Initial Population

Each individual (chromosome) is a **permutation** of all shelf slot indices.
Position `i` in the chromosome means "SKU *i* is stored in the slot indicated
by `chromosome[i]`".  Random permutations give the GA a diverse starting point.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.1  Initial Population Generator
# ─────────────────────────────────────────────────────────────────────────────

def initial_population(pop_size: int) -> List[List[int]]:
    """Generate an initial population of random SKU-to-location assignments.

    Each chromosome is a permutation of all slot indices (0 … N-1).
    The interpretation is: chromosome[slot] = SKU stored at that slot,
    so slot *i* holds SKU chromosome[i].

    Parameters
    ----------
    pop_size : Number of chromosomes in the population.

    Returns
    -------
    List of `pop_size` unique-permutation chromosomes.
    """
    n_slots = len(sku_location_3d)
    return [random.sample(range(n_slots), n_slots) for _ in range(pop_size)]


### 6.2 Crossover — Order Crossover (OX)

We use **Order Crossover (OX)**, which is well-suited to permutation-encoded
problems because it preserves the *relative order* of elements from both parents,
guaranteeing that every child is a valid permutation (no repeated or missing genes).

**Algorithm:**
1. Pick a random contiguous segment `[start, end)` from **Parent 1**.
2. Copy that segment directly into the child.
3. Fill the remaining positions (left to right, wrapping) with genes from  
   **Parent 2**, in the order they appear, skipping any already in the child.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2  Order Crossover (OX)
# ─────────────────────────────────────────────────────────────────────────────

def order_crossover(parent1: List[int], parent2: List[int]) -> List[int]:
    """Produce a child chromosome via Order Crossover (OX).

    OX guarantees the result is a valid permutation because:
      - The copied segment from parent1 contributes no duplicates.
      - Remaining genes are drawn from parent2 *excluding* those already present.

    Parameters
    ----------
    parent1 : First parent chromosome (permutation of slot indices).
    parent2 : Second parent chromosome (permutation of slot indices).

    Returns
    -------
    Child chromosome — a valid permutation of the same length.
    """
    n = len(parent1)

    # Randomly select a contiguous gene segment from parent1
    start = np.random.randint(0, n)
    end = np.random.randint(start + 1, n + 1)   # end is exclusive

    # Initialise child with sentinel value -1 (unfilled positions)
    child = [-1] * n

    # Copy the selected segment from parent1 directly into the child
    for i in range(start, end):
        child[i] = parent1[i]

    # Fill remaining positions from parent2, preserving relative order
    # and skipping genes already present in the child
    remaining = [gene for gene in parent2 if gene not in child]
    fill_idx = 0
    for i in range(n):
        if child[i] == -1:
            child[i] = remaining[fill_idx]
            fill_idx += 1

    return child


### 6.3 Mutation — Swap Mutation

**Swap mutation** randomly selects two positions in the chromosome and exchanges
their genes.  This small perturbation prevents premature convergence while
preserving the permutation validity constraint.

The mutation is applied stochastically: only chromosomes that pass the
`mutation_rate` threshold are mutated.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3  Swap Mutation
# ─────────────────────────────────────────────────────────────────────────────

def swap_mutation(individual: List[int], mutation_rate: float) -> List[int]:
    """Apply swap mutation to a chromosome with the given probability.

    Two randomly chosen genes are swapped.  Because we swap entire genes
    (slot indices), the chromosome remains a valid permutation after mutation.

    Parameters
    ----------
    individual    : Chromosome to (possibly) mutate.
    mutation_rate : Probability in [0, 1] that mutation occurs.

    Returns
    -------
    The (possibly mutated) chromosome.
    """
    if np.random.rand() < mutation_rate:
        # Choose two distinct positions
        idx1, idx2 = np.random.choice(len(individual), size=2, replace=False)
        # Swap in-place
        individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual


## 7. Main Genetic Algorithm Loop

The GA follows a **steady-state generational** scheme with **top-20 elitism**:

```
Initialise population (pop_size random chromosomes)
For each generation:
    Evaluate fitness of every chromosome
    Keep best 20 chromosomes unchanged (elitism)
    Generate pop_size new children via OX crossover from elite parents
    Apply swap mutation to all chromosomes
    Replace population with elite + mutated offspring
    Track best solution seen across all generations
```

**Key design choices:**
- **Elitism ratio**: top 20 / pop_size ≈ 10% — balances exploitation vs exploration  
- **Parent pool**: crossover always samples from the elite 20 — directs search  
- **Mutation on elite**: the 20 elite chromosomes are also subject to mutation,  
  preventing exact clones from dominating the pool


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.1  Genetic Algorithm — Main Entry Point
# ─────────────────────────────────────────────────────────────────────────────

def genetic_algorithm(
    pop_size: int = DEFAULT_POP_SIZE,
    generations: int = DEFAULT_GENERATIONS,
    mutation_rate: float = DEFAULT_MUTATION_RATE,
    verbose: bool = True,
) -> Tuple[List[int], float, List[float]]:
    """Run the Genetic Algorithm for warehouse SKU assignment optimisation.

    Parameters
    ----------
    pop_size      : Number of chromosomes in the population.
    generations   : Number of generations to evolve.
    mutation_rate : Probability of swap mutation per chromosome per generation.
    verbose       : If True, print improvement messages and separator lines.

    Returns
    -------
    best_route    : The chromosome (SKU permutation) with the lowest fitness seen.
    best_distance : The corresponding fitness value (distance + time, in seconds/metres).
    history       : List of best fitness values, one per generation (for plotting).
    """
    # ── Initialisation ─────────────────────────────────────────────────────
    population = initial_population(pop_size)
    best_route: Optional[List[int]] = None
    best_distance: float = float("inf")
    history: List[float] = []          # tracks best fitness per generation

    for gen in range(generations):
        if verbose:
            print(f"\n── Generation {gen + 1}/{generations} ──────────────────────")

        # ── Fitness Evaluation + Crossover ─────────────────────────────────
        fitness_values, offspring = evaluate_population(population)

        # ── Elitism: track global best ─────────────────────────────────────
        gen_best_fitness = min(fitness_values)
        if gen_best_fitness < best_distance:
            best_distance = gen_best_fitness
            best_route = population[int(np.argmin(fitness_values))]
            if verbose:
                print(f"  ★ Improved! Best fitness = {best_distance:.2f}")
                # print(f"  Best route = {best_route}")   # uncomment if needed

        history.append(best_distance)

        # ── Mutation ───────────────────────────────────────────────────────
        # Apply swap mutation to every chromosome in the offspring population
        population = [swap_mutation(ind, mutation_rate) for ind in offspring]

    if verbose:
        print(f"\n✓ GA complete — Best fitness: {best_distance:.2f}")

    return best_route, best_distance, history


## 8. Experimental Results

### 8.1 Baseline Run

Run the GA with default parameters and plot convergence.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.1  Baseline GA Run
# ─────────────────────────────────────────────────────────────────────────────
# Parameters: pop=200, generations=200, mutation_rate=0.5, seed=30

random.seed(30)
np.random.seed(30)

best_route, best_distance, history = genetic_algorithm(
    pop_size=DEFAULT_POP_SIZE,
    generations=DEFAULT_GENERATIONS,
    mutation_rate=DEFAULT_MUTATION_RATE,
    verbose=True,
)

# ── Convergence Plot ──────────────────────────────────────────────────────────
plt.figure(figsize=(12, 5))
plt.plot(history, linewidth=2, color="steelblue")
plt.xlabel("Generation")
plt.ylabel("Best Fitness (distance + retrieval time)")
plt.title("GA Convergence — Baseline Run (pop=200, μ=0.5, seed=30)")
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\nFinal best distance : {best_distance:.2f}")
print(f"Best route (first 20 slots): {best_route[:20]} ...")


### 8.2 Sensitivity Analysis — Population Size

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.2  Population Size Sensitivity
# ─────────────────────────────────────────────────────────────────────────────
# We run the GA for 10 generations across 5 population sizes to understand
# the trade-off between population diversity and computational cost.
# ─────────────────────────────────────────────────────────────────────────────

QUICK_GENERATIONS = 10     # short run for sensitivity sweep
POP_SIZES = range(100, 501, 100)

plt.figure(figsize=(18, 7))

for pop_size in POP_SIZES:
    random.seed(30)
    np.random.seed(30)
    _, _, bd = genetic_algorithm(
        pop_size=pop_size,
        generations=QUICK_GENERATIONS,
        mutation_rate=0.1,
        verbose=False,
    )
    plt.plot(range(QUICK_GENERATIONS), bd,
             label=f"pop={pop_size}", linestyle="-", marker="o", markersize=4)

plt.xlabel("Generation")
plt.ylabel("Best Fitness")
plt.title("Population Size Sensitivity (10 Generations, μ=0.1, seed=30)")
plt.legend(title="Population Size")
plt.grid(True)
plt.tight_layout()
plt.show()


### 8.3 Sensitivity Analysis — Mutation Rate

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.3  Mutation Rate Sensitivity
# ─────────────────────────────────────────────────────────────────────────────
# Each curve shows how quickly the GA converges under a different mutation rate.
# Very low rates → slow exploration; very high rates → destroys good solutions.
# ─────────────────────────────────────────────────────────────────────────────

SENSITIVITY_GENERATIONS = 100
MUTATION_RATES = np.arange(0.0, 1.0, 0.1)

plt.figure(figsize=(18, 7))

for mu in MUTATION_RATES:
    random.seed(30)
    np.random.seed(30)
    _, _, bd = genetic_algorithm(
        pop_size=200,
        generations=SENSITIVITY_GENERATIONS,
        mutation_rate=mu,
        verbose=False,
    )
    plt.plot(range(SENSITIVITY_GENERATIONS), bd,
             label=f"μ={mu:.1f}", linestyle="-", marker="o", markersize=3)

plt.xlabel("Generation")
plt.ylabel("Best Fitness")
plt.title("Mutation Rate Sensitivity (100 Generations, pop=200, seed=30)")
plt.legend(title="Mutation Rate", ncol=2)
plt.grid(True)
plt.tight_layout()
plt.show()


## 9. Visualisation Utilities

### 9.1 Excel Heat-Map Visualisation

The following utility exports a chromosome (SKU assignment) to a colour-coded
Excel workbook.  Each cell represents a shelf slot and is coloured by the SKU
range stored there:

| Colour | SKU range | Interpretation |
|---|---|---|
| 🟩 Green | 0–99 | Top-100 fast movers |
| 🔵 Light blue | 100–199 | Medium-velocity |
| 💙 Blue | 200–299 | Lower-medium velocity |
| 🔴 Red | 300–499 | Slow movers |

This lets researchers visually inspect whether high-demand SKUs cluster near
the depot (front of the warehouse) after optimisation.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.1  Excel Heat-Map Export
# ─────────────────────────────────────────────────────────────────────────────

# Colour fills — defined once and reused across all exports
_FILL_GREEN      = PatternFill(start_color="00FF00", end_color="00FF00", fill_type="solid")
_FILL_LIGHT_BLUE = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")
_FILL_BLUE       = PatternFill(start_color="0000FF", end_color="0000FF", fill_type="solid")
_FILL_RED        = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")


def _sku_fill(sku_id: int) -> Optional[PatternFill]:
    """Return the openpyxl PatternFill corresponding to a SKU's demand tier."""
    if 0 <= sku_id <= 99:
        return _FILL_GREEN
    elif 100 <= sku_id <= 199:
        return _FILL_LIGHT_BLUE
    elif 200 <= sku_id <= 299:
        return _FILL_BLUE
    elif 300 <= sku_id <= 499:
        return _FILL_RED
    return None


def export_route_to_excel(
    numbers: List[int],
    filepath: str,
    block_cols: int = 4,
    block_rows: int = 25,
    blocks_per_row: int = 4,
) -> None:
    """Write a colour-coded SKU assignment to an Excel workbook.

    The route (list of SKU IDs) is laid out in rectangular blocks.  Each block
    represents one rack face.  Cells are coloured by SKU demand tier.

    Parameters
    ----------
    numbers       : Ordered list of SKU IDs (the chromosome / result list).
    filepath      : Output .xlsx file path.
    block_cols    : Number of Excel columns per rack block.
    block_rows    : Number of Excel rows per rack block.
    blocks_per_row: How many blocks to place side-by-side before wrapping.
    """
    wb = Workbook()
    sheet = wb.active
    numbers_per_block = block_cols * block_rows

    for block_offset in range(0, len(numbers), numbers_per_block):
        block_num = block_offset // numbers_per_block
        # Top-left corner of this block in the spreadsheet
        start_row = (block_num // blocks_per_row) * block_rows + 1
        start_col = (block_num % blocks_per_row) * block_cols + 1

        for row_i in range(block_rows):
            for col_j in range(block_cols):
                number_idx = block_offset + row_i * block_cols + col_j
                if number_idx >= len(numbers):
                    break
                sku_id = numbers[number_idx]
                cell = sheet.cell(
                    row=start_row + row_i,
                    column=start_col + col_j,
                    value=sku_id,
                )
                fill = _sku_fill(sku_id)
                if fill:
                    cell.fill = fill

    wb.save(filepath)
    print(f"Saved: {filepath}")


# ── Example export using the W-routing result ─────────────────────────────────
# (Replace 'best_route' with any chromosome or pre-computed result list)
w_result = [321, 7, 3, 320, 266, 27, 54, 328, 278, 5,
            66, 405, 128, 38, 43, 410, 263, 21, 11, 325]   # truncated sample

export_route_to_excel(
    numbers=w_result,
    filepath="output_w_routing_sample.xlsx",
    block_cols=4,
    block_rows=25,
    blocks_per_row=2,
)


### 9.2 SKU Frequency Table

For quick inspection without opening Excel, render a DataFrame showing which
slot each SKU maps to in the best chromosome.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9.2  Best-Route Summary Table
# ─────────────────────────────────────────────────────────────────────────────

def route_summary_dataframe(
    chromosome: List[int],
    n_cols: int = 8,
) -> pd.DataFrame:
    """Build a DataFrame representing the warehouse shelf layout for a chromosome.

    Shelf slots are arranged in a grid with `n_cols` columns, filled from the
    bottom-left corner upward (row 0 = bottom shelf, row N = top).

    Parameters
    ----------
    chromosome : Chromosome mapping slot → SKU.
    n_cols     : Number of columns in the display table.

    Returns
    -------
    DataFrame where each cell contains the SKU stored in that slot.
    """
    n_rows = (len(chromosome) + n_cols - 1) // n_cols
    df = pd.DataFrame(index=range(n_rows), columns=range(n_cols))

    # Fill from bottom-right to top-left for intuitive warehouse orientation
    rev = chromosome[::-1]
    flat_idx = 0
    for col in range(n_cols):
        for row in range(n_rows - 1, -1, -1):
            if flat_idx < len(rev):
                df.iat[row, col] = rev[flat_idx]
                flat_idx += 1

    return df


# Display the first 40 slots of the best route as a shelf map
if best_route is not None:
    shelf_map = route_summary_dataframe(best_route[:40], n_cols=8)
    print("Shelf map (first 40 slots):")
    print(shelf_map.to_string())

    # Optionally export to CSV
    shelf_map.to_csv("best_route_shelf_map.csv", index=False)
    print("\nExported to best_route_shelf_map.csv")


## 10. Recorded Experimental Results

The following cells preserve the best routes found in prior long-running
experiments (1 000 orders, 1 500 generations).  These are stored as reference
values for comparison and visualisation without re-running the full algorithm.

> **Seeds used**: 30, 88, 5 (see individual result blocks below).


### 10.1 W-Policy + S-Shape Routing (seed=30, 1 000 orders, 1 500 generations)

In [ ]:
# Best route found under W-policy with S-shape routing
result_w_sshape_seed30 = [
    321, 7, 3, 320, 266, 27, 54, 328, 278, 5, 66, 405, 128, 38, 43, 410,
    263, 21, 11, 325, 297, 82, 184, 326, 203, 44, 153, 436, 353, 6, 194, 391,
    231, 14, 167, 438, 309, 42, 60, 427, 242, 32, 65, 390, 373, 134, 67, 432,
    136, 122, 12, 382, 300, 58, 170, 426, 233, 115, 94, 493,
    # ... (truncated for display — see full list in original research data)
]
print(f"Route length: {len(result_w_sshape_seed30)} (truncated sample)")


### 10.2 GZA + S-Shape Routing (seed=45, 1 000 orders, 1 500 generations)

In [ ]:
# Best route found under GZA-policy with S-shape routing
# Best Distance: 1 829 170.0
result_gza_sshape_seed45 = [
    4, 22, 138, 33, 40, 2, 1, 6, 3, 72, 107, 146, 159, 71, 58, 166,
    84, 113, 158, 82, 74, 220, 221, 167, 129, 235, 255, 101, 164, 133,
    # ... (truncated for display)
]
BEST_DISTANCE_GZA_SSHAPE_S45 = 1_829_170.0

print(f"GZA + S-Shape best distance : {BEST_DISTANCE_GZA_SSHAPE_S45:,.0f}")
print(f"Route sample (first 20 slots): {result_gza_sshape_seed45[:20]}")


## 11. Summary & Next Steps

### Key Findings
- The **GZA (Gold-Zone Assignment)** storage policy consistently outperforms
  random W-assignment by directing high-demand SKUs to ergonomic shelf levels,
  reducing both travel distance and vertical retrieval time.
- **S-shape routing** provides a good baseline; return and mid-point routing
  offer further improvements for specific order-density profiles.
- The GA converges reliably within ~200 generations for populations of 200.

### Recommended Extensions
1. **Inver-Over crossover** — a stronger permutation operator that may accelerate
   convergence; scaffolded code exists in the research notes.
2. **Simulated Annealing (SA) hybrid** — combine SA local search with the GA
   global search to escape local optima.
3. **Multi-objective optimisation** — jointly minimise distance and maximise
   ergonomic score (reduce picks at difficult levels).
4. **Real warehouse data** — replace the simulated demand model with actual
   WMS order data.
